In [3]:
import json
from pydantic import BaseModel, Field, field_validator
import pandas as pd

In [2]:
import pytest, os

In [53]:
df = pd.read_json('orders.json')

In [54]:
df.head()

,sales,type,market,date order,date shipping,id,status,quantity,loc,product,customer,order zipcode
0,327.75,DEBIT,Pacific Asia,2018-01-31T22:56:00,2018-02-03T22:56:00,77202,COMPLETE,1,"{'country': 'Indonesia', 'state': 'Java Occide...","{'name': 'Smart watch ', 'category_id': 73, 'c...","{'id': 20755, 'firstname': 'Cally', 'lastname'...",NaN
1,327.75,TRANSFER,Pacific Asia,2018-01-13T12:27:00,2018-01-18T12:27:00,75939,PENDING,1,"{'country': 'India', 'state': 'Rajastán', 'reg...","{'name': 'Smart watch ', 'category_id': 73, 'c...","{'id': 19492, 'firstname': 'Irene', 'lastname'...",NaN
2,327.75,CASH,Pacific Asia,2018-01-13T12:06:00,2018-01-17T12:06:00,75938,CLOSED,1,"{'country': 'India', 'state': 'Rajastán', 'reg...","{'name': 'Smart watch ', 'category_id': 73, 'c...","{'id': 19491, 'firstname': 'Gillian', 'lastnam...",NaN
3,327.75,DEBIT,Pacific Asia,2018-01-13T11:45:00,2018-01-16T11:45:00,75937,COMPLETE,1,"{'country': 'Australia', 'state': 'Queensland'...","{'name': 'Smart watch ', 'category_id': 73, 'c...","{'id': 19490, 'firstname': 'Tana', 'lastname':...",NaN
4,327.75,PAYMENT,Pacific Asia,2018-01-13T11:24:00,2018-01-15T11:24:00,75936,PENDING_PAYMENT,1,"{'country': 'Australia', 'state': 'Queensland'...","{'name': 'Smart watch ', 'category_id': 73, 'c...","{'id': 19489, 'firstname': 'Orli', 'lastname':...",NaN


In [47]:
df.describe().style.format({"sales": "{:.2f} €", "quantity": "{:.0f}","id": "{:.0f}"})

,sales,id,quantity,order zipcode
count,180519.00 €,180519,180519,24840.000000
mean,203.77 €,36222,2,55426.132327
std,132.27 €,21045,1,31919.279101
min,9.99 €,1,1,1040.000000
25%,119.98 €,18057,1,23464.000000
50%,199.92 €,36140,1,59405.000000
75%,299.95 €,54144,3,90008.000000
max,1999.99 €,77204,5,99301.000000


In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 12 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   sales          180519 non-null  float64
 1   type           180519 non-null  str    
 2   market         180519 non-null  str    
 3   date order     180519 non-null  str    
 4   date shipping  180519 non-null  str    
 5   id             180519 non-null  int64  
 6   status         180519 non-null  str    
 7   quantity       180519 non-null  int64  
 8   loc            180519 non-null  object 
 9   product        180519 non-null  object 
 10  customer       180519 non-null  object 
 11  order zipcode  24840 non-null   float64
dtypes: float64(2), int64(2), object(3), str(5)
memory usage: 16.5+ MB


In [37]:
from pandas.io.formats.style import Styler
df.head().style.format({"sales": "{:.2f} €", "quantity": "{:,d}"})

,sales,type,market,date order,date shipping,id,status,quantity,loc,product,customer,order zipcode
0,327.75 €,DEBIT,Pacific Asia,2018-01-31T22:56:00,2018-02-03T22:56:00,77202,COMPLETE,1,"{'country': 'Indonesia', 'state': 'Java Occidental', 'region': 'Southeast Asia', 'city': 'Bekasi'}","{'name': 'Smart watch ', 'category_id': 73, 'category_name': 'Sporting Goods', 'price': 327.75}","{'id': 20755, 'firstname': 'Cally', 'lastname': 'Holloway', 'loc': {'country': 'Puerto Rico', 'state': 'PR', 'city': 'Caguas', 'zipcode': 725.0, 'street': '5365 Noble Nectar Island'}}",nan
1,327.75 €,TRANSFER,Pacific Asia,2018-01-13T12:27:00,2018-01-18T12:27:00,75939,PENDING,1,"{'country': 'India', 'state': 'Rajastán', 'region': 'South Asia', 'city': 'Bikaner'}","{'name': 'Smart watch ', 'category_id': 73, 'category_name': 'Sporting Goods', 'price': 327.75}","{'id': 19492, 'firstname': 'Irene', 'lastname': 'Luna', 'loc': {'country': 'Puerto Rico', 'state': 'PR', 'city': 'Caguas', 'zipcode': 725.0, 'street': '2679 Rustic Loop'}}",nan
2,327.75 €,CASH,Pacific Asia,2018-01-13T12:06:00,2018-01-17T12:06:00,75938,CLOSED,1,"{'country': 'India', 'state': 'Rajastán', 'region': 'South Asia', 'city': 'Bikaner'}","{'name': 'Smart watch ', 'category_id': 73, 'category_name': 'Sporting Goods', 'price': 327.75}","{'id': 19491, 'firstname': 'Gillian', 'lastname': 'Maldonado', 'loc': {'country': 'EE. UU.', 'state': 'CA', 'city': 'San Jose', 'zipcode': 95125.0, 'street': '8510 Round Bear Gate'}}",nan
3,327.75 €,DEBIT,Pacific Asia,2018-01-13T11:45:00,2018-01-16T11:45:00,75937,COMPLETE,1,"{'country': 'Australia', 'state': 'Queensland', 'region': 'Oceania', 'city': 'Townsville'}","{'name': 'Smart watch ', 'category_id': 73, 'category_name': 'Sporting Goods', 'price': 327.75}","{'id': 19490, 'firstname': 'Tana', 'lastname': 'Tate', 'loc': {'country': 'EE. UU.', 'state': 'CA', 'city': 'Los Angeles', 'zipcode': 90027.0, 'street': '3200 Amber Bend'}}",nan
4,327.75 €,PAYMENT,Pacific Asia,2018-01-13T11:24:00,2018-01-15T11:24:00,75936,PENDING_PAYMENT,1,"{'country': 'Australia', 'state': 'Queensland', 'region': 'Oceania', 'city': 'Townsville'}","{'name': 'Smart watch ', 'category_id': 73, 'category_name': 'Sporting Goods', 'price': 327.75}","{'id': 19489, 'firstname': 'Orli', 'lastname': 'Hendricks', 'loc': {'country': 'Puerto Rico', 'state': 'PR', 'city': 'Caguas', 'zipcode': 725.0, 'street': '8671 Iron Anchor Corners'}}",nan


In [34]:
df.head()

,sales,type,market,date order,date shipping,id,status,quantity,loc,product,customer,order zipcode
0,327.75,DEBIT,Pacific Asia,2018-01-31T22:56:00,2018-02-03T22:56:00,77202,COMPLETE,1,"{'country': 'Indonesia', 'state': 'Java Occide...","{'name': 'Smart watch ', 'category_id': 73, 'c...","{'id': 20755, 'firstname': 'Cally', 'lastname'...",NaN
1,327.75,TRANSFER,Pacific Asia,2018-01-13T12:27:00,2018-01-18T12:27:00,75939,PENDING,1,"{'country': 'India', 'state': 'Rajastán', 'reg...","{'name': 'Smart watch ', 'category_id': 73, 'c...","{'id': 19492, 'firstname': 'Irene', 'lastname'...",NaN
2,327.75,CASH,Pacific Asia,2018-01-13T12:06:00,2018-01-17T12:06:00,75938,CLOSED,1,"{'country': 'India', 'state': 'Rajastán', 'reg...","{'name': 'Smart watch ', 'category_id': 73, 'c...","{'id': 19491, 'firstname': 'Gillian', 'lastnam...",NaN
3,327.75,DEBIT,Pacific Asia,2018-01-13T11:45:00,2018-01-16T11:45:00,75937,COMPLETE,1,"{'country': 'Australia', 'state': 'Queensland'...","{'name': 'Smart watch ', 'category_id': 73, 'c...","{'id': 19490, 'firstname': 'Tana', 'lastname':...",NaN
4,327.75,PAYMENT,Pacific Asia,2018-01-13T11:24:00,2018-01-15T11:24:00,75936,PENDING_PAYMENT,1,"{'country': 'Australia', 'state': 'Queensland'...","{'name': 'Smart watch ', 'category_id': 73, 'c...","{'id': 19489, 'firstname': 'Orli', 'lastname':...",NaN


In [11]:
def flatten(d, parent_key="", sep="."):
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten(v, new_key, sep))
        else:
            items[new_key] = v
    return items

In [ ]:
df_flat = pd.json_normalize(df["loc"].apply(flatten))

In [57]:
df.to_dict(orient='records')

[{'sales': 327.75,
  'type': 'DEBIT',
  'market': 'Pacific Asia',
  'date order': '2018-01-31T22:56:00',
  'date shipping': '2018-02-03T22:56:00',
  'id': 77202,
  'status': 'COMPLETE',
  'quantity': 1,
  'loc': {'country': 'Indonesia',
   'state': 'Java Occidental',
   'region': 'Southeast Asia',
   'city': 'Bekasi'},
  'product': {'name': 'Smart watch ',
   'category_id': 73,
   'category_name': 'Sporting Goods',
   'price': 327.75},
  'customer': {'id': 20755,
   'firstname': 'Cally',
   'lastname': 'Holloway',
   'loc': {'country': 'Puerto Rico',
    'state': 'PR',
    'city': 'Caguas',
    'zipcode': 725.0,
    'street': '5365 Noble Nectar Island'}},
  'order zipcode': nan},
 {'sales': 327.75,
  'type': 'TRANSFER',
  'market': 'Pacific Asia',
  'date order': '2018-01-13T12:27:00',
  'date shipping': '2018-01-18T12:27:00',
  'id': 75939,
  'status': 'PENDING',
  'quantity': 1,
  'loc': {'country': 'India',
   'state': 'Rajastán',
   'region': 'South Asia',
   'city': 'Bikaner'},
  

In [66]:
pd.json_normalize(data=df.to_dict(orient='records'),sep='.')

,sales,type,market,date order,date shipping,id,status,quantity,order zipcode,loc.country,...,product.category_name,product.price,customer.id,customer.firstname,customer.lastname,customer.loc.country,customer.loc.state,customer.loc.city,customer.loc.zipcode,customer.loc.street
0,327.750000,DEBIT,Pacific Asia,2018-01-31T22:56:00,2018-02-03T22:56:00,77202,COMPLETE,1,NaN,Indonesia,...,Sporting Goods,327.750000,20755,Cally,Holloway,Puerto Rico,PR,Caguas,725.0,5365 Noble Nectar Island
1,327.750000,TRANSFER,Pacific Asia,2018-01-13T12:27:00,2018-01-18T12:27:00,75939,PENDING,1,NaN,India,...,Sporting Goods,327.750000,19492,Irene,Luna,Puerto Rico,PR,Caguas,725.0,2679 Rustic Loop
2,327.750000,CASH,Pacific Asia,2018-01-13T12:06:00,2018-01-17T12:06:00,75938,CLOSED,1,NaN,India,...,Sporting Goods,327.750000,19491,Gillian,Maldonado,EE. UU.,CA,San Jose,95125.0,8510 Round Bear Gate
3,327.750000,DEBIT,Pacific Asia,2018-01-13T11:45:00,2018-01-16T11:45:00,75937,COMPLETE,1,NaN,Australia,...,Sporting Goods,327.750000,19490,Tana,Tate,EE. UU.,CA,Los Angeles,90027.0,3200 Amber Bend
4,327.750000,PAYMENT,Pacific Asia,2018-01-13T11:24:00,2018-01-15T11:24:00,75936,PENDING_PAYMENT,1,NaN,Australia,...,Sporting Goods,327.750000,19489,Orli,Hendricks,Puerto Rico,PR,Caguas,725.0,8671 Iron Anchor Corners
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180514,399.980011,CASH,Pacific Asia,2016-01-16T03:40:00,2016-01-20T03:40:00,26043,CLOSED,1,NaN,China,...,Fishing,399.980011,1005,Maria,Peterson,EE. UU.,NY,Brooklyn,11207.0,1322 Broad Glade
180515,399.980011,DEBIT,Pacific Asia,2016-01-16T01:34:00,2016-01-19T01:34:00,26037,COMPLETE,1,NaN,Japón,...,Fishing,399.980011,9141,Ronald,Clark,EE. UU.,CA,Bakersfield,93304.0,7330 Broad Apple Moor
180516,399.980011,TRANSFER,Pacific Asia,2016-01-15T21:00:00,2016-01-20T21:00:00,26024,PENDING,1,NaN,Australia,...,Fishing,399.980011,291,John,Smith,EE. UU.,CT,Bristol,6010.0,97 Burning Landing
180517,399.980011,PAYMENT,Pacific Asia,2016-01-15T20:18:00,2016-01-18T20:18:00,26022,PENDING_PAYMENT,1,NaN,Australia,...,Fishing,399.980011,2813,Mary,Smith,Puerto Rico,PR,Caguas,725.0,2585 Silent Autumn Landing


In [18]:
pd.concat([df_flat,df["type"]],axis='columns')

,country,state,region,city,type
0,Indonesia,Java Occidental,Southeast Asia,Bekasi,DEBIT
1,India,Rajastán,South Asia,Bikaner,TRANSFER
2,India,Rajastán,South Asia,Bikaner,CASH
3,Australia,Queensland,Oceania,Townsville,DEBIT
4,Australia,Queensland,Oceania,Townsville,PAYMENT
...,...,...,...,...,...
180514,China,Shanghái,Eastern Asia,Shanghái,CASH
180515,Japón,Osaka,Eastern Asia,Hirakata,DEBIT
180516,Australia,Australia del Sur,Oceania,Adelaide,TRANSFER
180517,Australia,Australia del Sur,Oceania,Adelaide,PAYMENT


In [24]:
df_flat.join(df['type'])

,country,state,region,city,type
0,Indonesia,Java Occidental,Southeast Asia,Bekasi,DEBIT
1,India,Rajastán,South Asia,Bikaner,TRANSFER
2,India,Rajastán,South Asia,Bikaner,CASH
3,Australia,Queensland,Oceania,Townsville,DEBIT
4,Australia,Queensland,Oceania,Townsville,PAYMENT
...,...,...,...,...,...
180514,China,Shanghái,Eastern Asia,Shanghái,CASH
180515,Japón,Osaka,Eastern Asia,Hirakata,DEBIT
180516,Australia,Australia del Sur,Oceania,Adelaide,TRANSFER
180517,Australia,Australia del Sur,Oceania,Adelaide,PAYMENT


In [27]:
df['type'].to_frame().join(df_flat)

,type,country,state,region,city
0,DEBIT,Indonesia,Java Occidental,Southeast Asia,Bekasi
1,TRANSFER,India,Rajastán,South Asia,Bikaner
2,CASH,India,Rajastán,South Asia,Bikaner
3,DEBIT,Australia,Queensland,Oceania,Townsville
4,PAYMENT,Australia,Queensland,Oceania,Townsville
...,...,...,...,...,...
180514,CASH,China,Shanghái,Eastern Asia,Shanghái
180515,DEBIT,Japón,Osaka,Eastern Asia,Hirakata
180516,TRANSFER,Australia,Australia del Sur,Oceania,Adelaide
180517,PAYMENT,Australia,Australia del Sur,Oceania,Adelaide


In [ ]:
parser.add_argument(
    "--limit",
    type=int,          # convertit en entier → args.limit == 50
    default=10,         # valeur si l'option n'est pas fournie
    help="nombre d'éléments à traiter (défaut : 10)"
)

In [ ]:
valeur obligatoire

add_argument("--name") (sans default)

args.name existe seulement si fourni



In [ ]:
Choix restreints

choices=["dev", "prod"]

erreur si autre valeur

